# Prepare Raw Air Quality Data for Model Inference

This notebook processes raw air quality data into a format compatible with the trained Random Forest model. The output CSV can be directly used for prediction.


In [26]:
# Import and paths
import pandas as pd
import numpy as np
from pathlib import Path

In [32]:
BASE_DIR = Path("..")
RAW_DATA_DIR = BASE_DIR / "data" / "raw" / "Dataset" / "PRSA2017_Data" / "PRSA_Data_"
OUTPUT_DIR = BASE_DIR / "data" / "processed"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_csv(
    RAW_DATA_DIR / "PRSA_Data_Dongsi.csv"
)

raw_df.head()

,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,9.0,9.0,3.0,17.0,300.0,89.0,-0.5,1024.5,-21.4,0.0,NNW,5.7,Dongsi
1,2,2013,3,1,1,4.0,4.0,3.0,16.0,300.0,88.0,-0.7,1025.1,-22.1,0.0,NW,3.9,Dongsi
2,3,2013,3,1,2,7.0,7.0,NaN,17.0,300.0,60.0,-1.2,1025.3,-24.6,0.0,NNW,5.3,Dongsi
3,4,2013,3,1,3,3.0,3.0,5.0,18.0,NaN,NaN,-1.4,1026.2,-25.5,0.0,N,4.9,Dongsi
4,5,2013,3,1,4,3.0,3.0,7.0,NaN,200.0,84.0,-1.9,1027.1,-24.5,0.0,NNW,3.2,Dongsi


## Datetime Construction and Basic Cleaning

In [33]:
raw_df["datetime"] = pd.to_datetime(
    raw_df[["year", "month", "day", "hour"]]
)

raw_df = raw_df.set_index("datetime")
raw_df = raw_df.drop(columns=["No"])

## PM2.5 Lag Features

In [34]:
for lag in [1, 3, 6, 12, 24]:
    raw_df[f"PM2.5_lag_{lag}h"] = raw_df["PM2.5"].shift(lag)

## PM2.5 Rolling Statistics

In [35]:
for window in [3, 6, 12, 24]:
    raw_df[f"PM2.5_roll_mean_{window}h"] = raw_df["PM2.5"].rolling(window).mean()
    raw_df[f"PM2.5_roll_std_{window}h"] = raw_df["PM2.5"].rolling(window).std()

## Time-Based Features

In [36]:
raw_df["hour"] = raw_df.index.hour
raw_df["dayofweek"] = raw_df.index.dayofweek
raw_df["month"] = raw_df.index.month

## Cyclical Encoding of Time Features

In [37]:
raw_df["hour_sin"] = np.sin(2 * np.pi * raw_df["hour"] / 24)
raw_df["hour_cos"] = np.cos(2 * np.pi * raw_df["hour"] / 24)

raw_df["month_sin"] = np.sin(2 * np.pi * raw_df["month"] / 12)
raw_df["month_cos"] = np.cos(2 * np.pi * raw_df["month"] / 12)

## Wind Direction One-Hot Encoding

In [38]:
wd_dummies = pd.get_dummies(raw_df["wd"], prefix="wd")
raw_df = pd.concat([raw_df, wd_dummies], axis=1)

## Canonical Feature Columns (Model Schema)

In [ ]:
FEATURE_COLUMNS = [
    "PM10","SO2","NO2","CO","O3","TEMP","PRES","DEWP","RAIN","WSPM", "PM2.5_lag_1h","PM2.5_lag_3h","PM2.5_lag_6h","PM2.5_lag_12h","PM2.5_lag_24h", "PM2.5_roll_mean_3h","PM2.5_roll_std_3h", "PM2.5_roll_mean_6h","PM2.5_roll_std_6h",
    "PM2.5_roll_mean_12h","PM2.5_roll_std_12h", "PM2.5_roll_mean_24h","PM2.5_roll_std_24h", "hour","dayofweek","month", "hour_sin","hour_cos","month_sin","month_cos", "wd_ENE","wd_ESE","wd_N","wd_NE","wd_NNE","wd_NNW","wd_NW", "wd_S","wd_SE","wd_SSE","wd_SSW","wd_SW", "wd_W","wd_WNW","wd_WSW"
]

## Enforce Feature Alignment

In [ ]:
for col in FEATURE_COLUMNS:
    if col not in raw_df.columns:
        raw_df[col] = 0

## Final Feature Selection and Cleaning

In [40]:
processed_df = raw_df[FEATURE_COLUMNS].dropna()
processed_df.head(), processed_df.shape

(                     PM10   SO2   NO2     CO    O3  TEMP    PRES  DEWP  RAIN  \
 datetime                                                                       
 2013-03-02 00:00:00  11.0  11.0  45.0  500.0  52.0  -0.7  1033.0 -18.3   0.0   
 2013-03-02 01:00:00   6.0  11.0  43.0  500.0  53.0  -1.6  1033.2 -17.4   0.0   
 2013-03-02 02:00:00   6.0  10.0  36.0  400.0  61.0  -2.2  1032.9 -16.7   0.0   
 2013-03-02 03:00:00   6.0  11.0  41.0  500.0  46.0  -3.1  1032.6 -15.8   0.0   
 2013-03-02 04:00:00   6.0   9.0  31.0  400.0  60.0  -3.1  1032.7 -16.1   0.0   
 
                      WSPM  ...  wd_NNW  wd_NW   wd_S  wd_SE  wd_SSE  wd_SSW  \
 datetime                   ...                                                
 2013-03-02 00:00:00   0.6  ...   False  False  False  False   False   False   
 2013-03-02 01:00:00   0.0  ...   False  False  False  False   False   False   
 2013-03-02 02:00:00   0.0  ...   False  False  False  False   False   False   
 2013-03-02 03:00:00   0.0  ...

## Save Processed Dataset

In [42]:
output_path = OUTPUT_DIR / "dongsi_ready_for_prediction.csv"

processed_df.to_csv(output_path)

print("Inference-ready dataset saved:")
print(output_path)


Inference-ready dataset saved:
..\data\processed\dongsi_ready_for_prediction.csv
